In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [10]:
df = pd.read_excel('data/serie_cafe_cepea_mensal.xlsx')
df.rename(columns = {
    'Data': 'date',
    'À vista R$': 'price_brl',
    'À vista US$': 'price_usd'
}, inplace = True)
df["log_price_brl"] = np.log(df["price_brl"])

In [11]:
import yfinance as yf
import pandas as pd
import numpy as np

fx = yf.download("BRL=X", start="1996-01-01")
fx = fx[["Close"]]
fx.rename(columns={"Close": "usd_brl"}, inplace=True)
fx.columns = fx.columns.droplevel(0)


fx = fx.reset_index()

[*********************100%***********************]  1 of 1 completed


In [12]:
cambio_df = fx

cambio_df['Day'] = cambio_df['Date'].dt.day
cambio_df['Month'] = cambio_df['Date'].dt.month
cambio_df['Year'] = cambio_df['Date'].dt.year


In [13]:
last_cambio = cambio_df.loc[
    cambio_df.groupby(['Year', 'Month'])['Day'].transform('max')
    == cambio_df['Day']
]

In [14]:
df['Day'] = df['date'].dt.day
df['Month'] = df['date'].dt.month
df['Year'] = df['date'].dt.year

In [15]:
df = pd.merge(
    df, last_cambio[['Year', 'Month', 'BRL=X']], on=['Year', 'Month'], how='left'
).rename(columns = {
    'BRL=X': 'usd_brl'
})

In [16]:
df["log_usd_brl"] = np.log(df["usd_brl"])

In [22]:
def create_multivariate_lags(train_df, n_lags=6):

    data = train_df.copy()

    for lag in range(1, n_lags + 1):
        data[f"price_lag{lag}"] = data["log_price_brl"].shift(lag)
        data[f"fx_lag{lag}"] = data["log_usd_brl"].shift(lag)

    data = data.dropna()

    X = data[[col for col in data.columns if "lag" in col]]
    y = data["log_price_brl"]

    return X, y

In [23]:
from xgboost import XGBRegressor
import numpy as np

def xgb_multivariate_model(train_df, horizon, n_lags=6):

    X_train, y_train = create_multivariate_lags(train_df, n_lags)

    model = XGBRegressor(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        objective="reg:squarederror"
    )

    model.fit(X_train, y_train)

    # construir última janela conhecida
    last_data = train_df.iloc[-n_lags:]

    features = []
    for lag in range(1, n_lags + 1):
        features.append(last_data["log_price_brl"].iloc[-lag])
        features.append(last_data["log_usd_brl"].iloc[-lag])

    X_pred = np.array(features).reshape(1, -1)

    # previsão recursiva
    for _ in range(horizon):
        y_hat = model.predict(X_pred)[0]
        X_pred = np.roll(X_pred, 2)
        X_pred[0, 0] = y_hat

    return y_hat

In [24]:
def walk_forward_forecast_multivariate(
    df,
    model_func,
    start_train_idx,
    horizon,
    n_lags=12
):

    forecasts = []
    actuals = []
    dates = []

    start = max(start_train_idx, n_lags)

    for t in range(start, len(df) - horizon):

        train_df = df.iloc[:t]

        y_hat = model_func(train_df, horizon, n_lags)

        actual = df["log_price_brl"].iloc[t + horizon]

        forecasts.append(y_hat)
        actuals.append(actual)
        dates.append(df.index[t + horizon])

    return pd.DataFrame({
        "y_true": actuals,
        "y_pred": forecasts
    }, index=dates)

In [25]:
def forecast_metrics(df):
    rmse = np.sqrt(np.mean((df.y_true - df.y_pred) ** 2))
    mae = np.mean(np.abs(df.y_true - df.y_pred))
    mape = np.mean(np.abs((df.y_true - df.y_pred) / df.y_true)) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape
    }

In [26]:
start_train_idx = df.index.get_indexer([pd.Timestamp("2018-01-01")])[0]

for h in [1, 3, 6]:

    print(f"\nRodando XGBoost MULTIVARIADO – horizonte {h}")

    results = walk_forward_forecast_multivariate(
        df[["log_price_brl", "log_usd_brl"]],
        xgb_multivariate_model,
        start_train_idx,
        horizon=h,
        n_lags=12
    )

    print(forecast_metrics(results))


Rodando XGBoost MULTIVARIADO – horizonte 1
{'RMSE': np.float64(3.0360770697034747), 'MAE': np.float64(1.823591523642269), 'MAPE': np.float64(34.84560642954307)}

Rodando XGBoost MULTIVARIADO – horizonte 3
{'RMSE': np.float64(3.048876096304264), 'MAE': np.float64(1.862654516814752), 'MAPE': np.float64(35.46521068952875)}

Rodando XGBoost MULTIVARIADO – horizonte 6
{'RMSE': np.float64(3.067081725730191), 'MAE': np.float64(1.918551631175372), 'MAPE': np.float64(36.36793246788886)}
